In [100]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import os

In [101]:
path = "Revised Project Customer Dataset 1.xlsx"
df = pd.read_excel(path)

In [102]:
df.columns = (
    df.columns
      .str.strip()
)

if "umber Pets" in df.columns:
    df.rename(columns={"umber Pets": "Number Pets"}, inplace=True)


In [103]:
df.head()

,CustomerID,Region,Gender,Age,Education Years,Employment Years,Job Category,Retired,Marital Status,Household Size,...,Coupon Redemption,Brand Tenure Months,Monthly Spend ProductA,Cumulative Spend ProductA,Monthly Spend ProductB,Cumulative Spend ProductB,Monthly Spend ProductC,Cumulative Spend ProductC,Total Avg Monthly Spend,High Value Customer
0,0002-GTOKLU-YVY,4,Female,32,19,34,Labor,No,Unmarried,4,...,1,51,552.24,84.92,303.04,4659.95,343.64,3037.73,84.37,0
1,0003-RLTRGE-IW2,3,Female,54,13,41,Agriculture,No,Unmarried,2,...,1,35,275.63,304.28,229.48,1662.20,717.79,1093.24,216.76,0
2,0003-UTGKPR-PRU,3,Female,77,18,2,Sales,No,Unmarried,2,...,0,2,307.43,703.88,0.93,1610.02,927.03,10979.31,125.57,1
3,0008-ZIQQOT-SGB,3,Male,44,22,37,Service,No,Unmarried,3,...,1,52,713.27,1.72,111.17,79.25,804.01,14886.62,777.17,0
4,0012-CIVYLF-839,4,Female,59,9,27,Agriculture,No,Married,5,...,1,42,421.92,495.2,173.55,606.26,701.66,967.23,1256.19,0


In [104]:

text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    if col in ["CustomerID"]:
        #don't modify the value for CustomerID
        continue
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.title()
    )



In [105]:
df.columns

Index(['CustomerID', 'Region', 'Gender', 'Age', 'Education Years',
       'Employment Years', 'Job Category', 'Retired', 'Marital Status',
       'Household Size', 'Household Income', 'Number Pets', 'Home Owner',
       'Car Ownership', 'Car Brand', 'Car Value', 'Commute Distance',
       'Political Party', 'Votes', 'Credit Card', 'Credit Card Tenure',
       'Active Lifestyle', 'TV Watching Hours', 'Streaming Svcs',
       'Wireless Internet', 'Smart Phone', 'Twitter Acct', 'LinkedIn Acct',
       'Facebook Acct', 'News Subscriber', 'Coupon Redemption',
       'Brand Tenure Months', 'Monthly Spend ProductA',
       'Cumulative Spend ProductA', 'Monthly Spend ProductB',
       'Cumulative Spend ProductB', 'Monthly Spend ProductC',
       'Cumulative Spend ProductC', 'Total Avg Monthly Spend',
       'High Value Customer'],
      dtype='object')

In [106]:
binary_map = {
    "Yes": 1, "No": 0,
    "Y": 1, "N": 0,
    "True": 1, "False": 0,
    "1": 1, "0": 0
}

binary_cols = [
    "Retired",
    "Home Owner",
    "Active Lifestyle",
    "Wireless Internet",
    "Smart Phone",
    "Twitter Acct",
    "LinkedIn Acct",
    "Facebook Acct",
    "News Subscriber",
    "Coupon Redemption",
    "High Value Customer",
    "Streaming Svcs",
    "Votes",
    "Political Party"
]

for col in binary_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .map(binary_map)
        )


In [107]:
currency_cols = [
    "Household Income",
    "Car Value",
    "Monthly Spend ProductA",
    "Cumulative Spend ProductA",
    "Monthly Spend ProductB",
    "Cumulative Spend ProductB",
    "Monthly Spend ProductC",
    "Cumulative Spend ProductC",
    "Total Avg Monthly Spend"
]

for col in currency_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(r"[$,]", "", regex=True)
        )

numeric_cols = currency_cols + [
    "Age",
    "Education Years",
    "Employment Years",
    "Household Size",
    "Number Pets",
    "Commute Distance"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [108]:

# Start with all rows assumed valid and an empty list of reasons per row
invalid_mask = pd.Series(False, index=df.index)
invalid_reasons = pd.Series([[] for _ in range(len(df))], index=df.index)

# Pattern: for each condition, build a boolean mask `cond`, OR it into `invalid_mask`,
# then append a human-readable reason to `invalid_reasons` for the same rows.

# 1) Nulls in critical fields -> mark invalid
cond = df.isna().any(axis=1)
invalid_mask |= cond
invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Null value'])

# 2) Ensure numeric columns are numeric (coercion created NaN for bad values) -> mark those as invalid
if 'numeric_cols' in locals():
    cond = df[numeric_cols].isna().any(axis=1)
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Invalid numeric value'])

# 3) Negative values for Education and Employment years are invalid
if 'Education Years' in df.columns:
    cond = df['Education Years'] < 0
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Negative Education Years'])
if 'Employment Years' in df.columns:
    cond = df['Employment Years'] < 0
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Negative Employment Years'])

# 4) Income must be >= 0
if 'Household Income' in df.columns:
    cond = df['Household Income'] < 0
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Negative Household Income'])

# 5) Household size must be > 0
if 'Household Size' in df.columns:
    cond = df['Household Size'] <= 0
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Non-positive Household Size'])

# 6) Education years must be less than Age - 5 (assuming people start school around age 5)
if set(['Education Years','Age']).issubset(df.columns):
    cond = ~(df['Education Years'] < df['Age'] - 5)
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Education Years >= Age - 5'])

# 6) Employment years cannot exceed plausible range: Employment Years <= Age - 14
# if set(['Employment Years','Age']).issubset(df.columns):
#     cond = ~(df['Employment Years'] <= (df['Age'] - 14))
#     invalid_mask |= cond
#     invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Employment Years implausible'])

# X) Credit card consistency: Credit Card == 0 but Credit Card Tenure > 0
if set(['Credit Card','Credit Card Tenure']).issubset(df.columns):
    cc = pd.to_numeric(df['Credit Card'], errors='coerce')
    cc_tenure = pd.to_numeric(df['Credit Card Tenure'], errors='coerce')
    cond = (cc == 0) & (cc_tenure > 0)
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Credit Card zero with positive tenure'])

# 7) Car value threshold: if provided and below 1000, mark invalid
if 'Car Value' in df.columns:
    # Coerce to numeric here to ensure values like 'Nocar' become NaN
    car_val = pd.to_numeric(df['Car Value'], errors='coerce')
    # Treat 0 as 'no car' (do not flag). Only flag positive values below threshold.
    cond = car_val.notna() & (car_val > 0) & (car_val < 1000)
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Car Value below threshold (non-zero)'])

# 9) Cumulative spend checks for Products A, B, C: cumulative >= monthly
for p in ['A','B','C']:
    monthly = f'Monthly Spend Product{p}'
    cumulative = f'Cumulative_Spend_Product{p}'
    if (monthly in df.columns) and (cumulative in df.columns):
        cond = df[cumulative].isna() | df[monthly].isna() | (df[cumulative] < df[monthly])
        invalid_mask |= cond
        invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + [f'Cumulative < Monthly for Product{p}'])

# 10) Remove duplicate customer records: keep first, mark subsequent duplicates as invalid
if 'CustomerID' in df.columns:
    dup_mask = df.duplicated(subset='CustomerID', keep='first')
    invalid_mask |= dup_mask
    invalid_reasons.loc[dup_mask] = invalid_reasons.loc[dup_mask].apply(lambda lst: lst + ['Duplicate CustomerID'])

# Build invalid and cleaned DataFrames
invalid_df = df[invalid_mask].copy()
# Attach a single string column summarizing reasons; empty string if none
invalid_df['Invalid Reasons'] = invalid_reasons[invalid_mask].apply(lambda lst: '; '.join(lst) if lst else '')
cleaned_df = df[~invalid_mask].copy()

# Ensure cleaned_df has no duplicates (keep first) and drop them if present
if 'CustomerID' in cleaned_df.columns:
    cleaned_df = cleaned_df.drop_duplicates(subset='CustomerID', keep='first')


In [109]:
cleaned_df.head()

,CustomerID,Region,Gender,Age,Education Years,Employment Years,Job Category,Retired,Marital Status,Household Size,...,Coupon Redemption,Brand Tenure Months,Monthly Spend ProductA,Cumulative Spend ProductA,Monthly Spend ProductB,Cumulative Spend ProductB,Monthly Spend ProductC,Cumulative Spend ProductC,Total Avg Monthly Spend,High Value Customer
0,0002-GTOKLU-YVY,4,Female,32,19,34,Labor,0,Unmarried,4,...,1,51,552.24,84.92,303.04,4659.95,343.64,3037.73,84.37,0
1,0003-RLTRGE-IW2,3,Female,54,13,41,Agriculture,0,Unmarried,2,...,1,35,275.63,304.28,229.48,1662.20,717.79,1093.24,216.76,0
2,0003-UTGKPR-PRU,3,Female,77,18,2,Sales,0,Unmarried,2,...,0,2,307.43,703.88,0.93,1610.02,927.03,10979.31,125.57,1
3,0008-ZIQQOT-SGB,3,Male,44,22,37,Service,0,Unmarried,3,...,1,52,713.27,1.72,111.17,79.25,804.01,14886.62,777.17,0
4,0012-CIVYLF-839,4,Female,59,9,27,Agriculture,0,Married,5,...,1,42,421.92,495.20,173.55,606.26,701.66,967.23,1256.19,0


In [110]:
invalid_df.head()

,CustomerID,Region,Gender,Age,Education Years,Employment Years,Job Category,Retired,Marital Status,Household Size,...,Brand Tenure Months,Monthly Spend ProductA,Cumulative Spend ProductA,Monthly Spend ProductB,Cumulative Spend ProductB,Monthly Spend ProductC,Cumulative Spend ProductC,Total Avg Monthly Spend,High Value Customer,Invalid Reasons
16,0026-GRFYZR-6M4,4,Male,19,15,34,Sales,0,Married,2,...,50,359.55,794.04,41.01,4115.56,866.86,6002.41,939.39,0,Education Years >= Age - 5
21,0031-EYRVUV-U32,3,Male,46,13,50,Agriculture,0,Married,2,...,1,474.62,228.20,25.52,867.30,306.92,3219.26,167.20,1,Car Value below threshold (non-zero)
36,0049-ZDULPT-RP6,2,Male,23,18,34,Agriculture,0,Married,2,...,65,513.63,1.80,300.14,1335.27,895.24,416.53,205.91,1,Education Years >= Age - 5
80,0119-ZUYERZ-58D,3,Male,21,21,10,Crafts,1,Married,6,...,62,704.41,998.84,37.35,610.08,305.31,4671.79,1236.12,0,Education Years >= Age - 5
90,0132-MBDKOW-6I6,4,Female,25,7,48,Service,0,Married,3,...,51,244.16,178.92,0.83,4077.38,272.11,12386.87,877.82,1,Car Value below threshold (non-zero)


In [111]:
# Create timestamped output directory: outputs/<yy-mm-dd-hh>/
ts = datetime.now().strftime('%y-%m-%d-%H')
outdir = Path('outputs') / ts
outdir.mkdir(parents=True, exist_ok=True)

In [112]:
# Save outputs as Excel workbooks
cleaned_df.to_excel(outdir / 'customer_data_cleaned.xlsx', index=False)
invalid_df.to_excel(outdir / 'invalid.xlsx', index=False)

print('Saved', cleaned_df.shape[0], 'clean rows and', invalid_df.shape[0], 'invalid rows to', outdir)

Saved 4695 clean rows and 305 invalid rows to outputs\26-04-16-21
